# 배우별 관객 수 연관 신호 분석

Lasso 회귀를 사용해 배우 출연 여부와 누적 관객 수의 연관 신호를 확인합니다. 단순 평균 비교가 아니라 기준선 모델과 테스트 성능을 함께 보며, 결과는 인과 효과가 아닌 데이터 내 연관성으로 해석합니다.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    return current.parent if current.name == "notebooks" else current


ROOT = find_project_root()


def configure_korean_font() -> None:
    preferred_fonts = ["AppleGothic", "NanumGothic", "NanumBarunGothic", "Malgun Gothic"]
    available_fonts = {font.name for font in fm.fontManager.ttflist}
    for family in preferred_fonts:
        if family in available_fonts:
            plt.rcParams["font.family"] = family
            break
    plt.rcParams["axes.unicode_minus"] = False


configure_korean_font()
ROOT

In [ ]:
DATA_CANDIDATES = [
    ROOT / "data/kobis_tmdb_title_matches.csv",
    ROOT / "data/tmdb_korean_movies_kobis.csv",
    Path("/content/drive/MyDrive/Colab Notebooks/top50_actors_boxoffice.csv"),
]
TITLE_COLUMNS = ["match_title", "title", "movie_name", "Movie Name"]
AUDIENCE_COLUMNS = ["audience_count", "kobis_audi_acc", "Audience", "Admissions"]
CAST_COLUMNS = ["tmdb_cast", "cast", "Actor"]


def read_first_available_csv(candidates):
    for path in candidates:
        if path.exists():
            return pd.read_csv(path, encoding="utf-8-sig"), path
    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"분석용 CSV를 찾지 못했습니다. 확인 경로:\n{searched}")


def first_existing_column(frame: pd.DataFrame, candidates, required: bool = True):
    for column in candidates:
        if column in frame.columns:
            return column
    if required:
        raise ValueError(f"필수 컬럼이 없습니다. 후보: {candidates}")
    return None


def clean_actor_name(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"\([^)]*\)$", "", text).strip()
    return text


def split_cast(value: object) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return []
    parts = text.split("|") if "|" in text else [text]
    names = [clean_actor_name(part) for part in parts]
    return [name for name in names if name]


def prepare_actor_movie_frame(frame: pd.DataFrame) -> pd.DataFrame:
    title_col = first_existing_column(frame, TITLE_COLUMNS, required=False)
    audience_col = first_existing_column(frame, AUDIENCE_COLUMNS)
    cast_col = first_existing_column(frame, CAST_COLUMNS)

    result = pd.DataFrame({
        "movie_title": frame[title_col] if title_col else [f"row_{idx}" for idx in range(len(frame))],
        "audience_count": pd.to_numeric(frame[audience_col], errors="coerce"),
        "actors": frame[cast_col].map(split_cast),
    })
    result = result.dropna(subset=["audience_count"])
    result = result[(result["audience_count"] > 0) & result["actors"].map(bool)].copy()
    result["audience_million"] = result["audience_count"] / 1_000_000
    return result.reset_index(drop=True)


def build_actor_matrix(movies: pd.DataFrame, min_movie_count: int = 3):
    actor_counts = pd.Series(
        [actor for actors in movies["actors"] for actor in actors],
        dtype="object",
    ).value_counts()
    selected_actors = actor_counts[actor_counts >= min_movie_count].index.tolist()
    if not selected_actors:
        raise ValueError("min_movie_count 기준을 만족하는 배우가 없습니다. 기준을 낮춰주세요.")

    matrix = pd.DataFrame(0, index=movies.index, columns=selected_actors, dtype=int)
    actor_set = set(selected_actors)
    for idx, actors in movies["actors"].items():
        for actor in actors:
            if actor in actor_set:
                matrix.at[idx, actor] = 1
    matrix = matrix.loc[:, matrix.sum().sort_values(ascending=False).index]
    return matrix, actor_counts


def actor_summary_table(movies: pd.DataFrame) -> pd.DataFrame:
    exploded = movies.explode("actors")
    return exploded.groupby("actors").agg(
        n_movies=("movie_title", "nunique"),
        mean_audience=("audience_count", "mean"),
        median_audience=("audience_count", "median"),
        hit_rate_3m=("audience_count", lambda values: (values >= 3_000_000).mean()),
        hit_rate_5m=("audience_count", lambda values: (values >= 5_000_000).mean()),
    )


raw_data, data_path = read_first_available_csv(DATA_CANDIDATES)
movies = prepare_actor_movie_frame(raw_data)
X, actor_counts = build_actor_matrix(movies, min_movie_count=3)
actor_stats = actor_summary_table(movies)

print(f"데이터: {data_path}")
print(f"영화 수: {len(movies):,}건")
print(f"분석 대상 배우 수: {X.shape[1]:,}명 (3편 이상 출연 기준)")
movies[["movie_title", "audience_count", "actors"]].head()

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


def regression_metrics(y_true_log, y_pred_log) -> dict[str, float]:
    actual = np.expm1(y_true_log)
    predicted = np.expm1(y_pred_log).clip(min=0)
    return {
        "MAE(명)": mean_absolute_error(actual, predicted),
        "RMSE(명)": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": r2_score(actual, predicted),
    }


y = np.log1p(movies["audience_count"])
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

baseline = DummyRegressor(strategy="mean")
baseline.fit(x_train, y_train)
baseline_pred = baseline.predict(x_test)

lasso = LassoCV(
    alphas=np.logspace(-4, 0, 80),
    cv=5,
    max_iter=20_000,
    random_state=RANDOM_STATE,
)
lasso.fit(x_train, y_train)
lasso_pred = lasso.predict(x_test)

metrics = pd.DataFrame([
    {"model": "평균 예측 기준선", **regression_metrics(y_test, baseline_pred)},
    {"model": "LassoCV 배우 원-핫", **regression_metrics(y_test, lasso_pred)},
])
metrics

In [ ]:
coef_summary = pd.DataFrame({
    "actor": X.columns,
    "lasso_coef_log": lasso.coef_,
}).join(actor_stats, on="actor")

baseline_log_audience = float(y_train.mean())
coef_summary["estimated_delta_audience"] = (
    np.expm1(baseline_log_audience + coef_summary["lasso_coef_log"]) - np.expm1(baseline_log_audience)
)
coef_summary["estimated_delta_million"] = coef_summary["estimated_delta_audience"] / 1_000_000
coef_summary["mean_audience_million"] = coef_summary["mean_audience"] / 1_000_000
coef_summary["median_audience_million"] = coef_summary["median_audience"] / 1_000_000

selected = coef_summary[coef_summary["lasso_coef_log"].abs() > 1e-8].copy()
if selected.empty:
    selected = coef_summary.reindex(coef_summary["lasso_coef_log"].abs().sort_values(ascending=False).index).head(20)

columns = [
    "actor", "n_movies", "lasso_coef_log", "estimated_delta_million",
    "mean_audience_million", "median_audience_million", "hit_rate_3m", "hit_rate_5m",
]
selected.sort_values("lasso_coef_log", ascending=False)[columns].head(20)

In [ ]:
plot_data = selected.sort_values("lasso_coef_log", ascending=False).head(15)
plot_data = plot_data.sort_values("estimated_delta_million")

fig, ax = plt.subplots(figsize=(10, 6))
colors = np.where(plot_data["estimated_delta_million"] >= 0, "#2a9d8f", "#e76f51")
ax.barh(plot_data["actor"], plot_data["estimated_delta_million"], color=colors)
ax.axvline(0, color="#333333", linewidth=1)
ax.set_title("Lasso가 선택한 배우별 관객 수 연관 신호")
ax.set_xlabel("평균 로그 관객 대비 추정 차이 (백만 명)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()

## 해석 메모

- Lasso 계수는 다른 배우 신호가 함께 들어간 상태에서 선택된 연관 신호입니다.
- 출연 편수가 적은 배우는 우연성이 커서 `n_movies`, 평균/중앙값 관객 수를 같이 확인해야 합니다.
- 제작비, 장르, 감독, 개봉 시기까지 통제하려면 `data/processed/movie_features.csv`의 메타 특성과 결합한 회귀 모델로 확장하는 것이 다음 단계입니다.